In [0]:
from pyspark.sql.functions import col

DELTA_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/"
GOLD_BASE_PATH = f"{DELTA_VOLUME_PATH}gold/"
CATALOG_NAME = "bc_transit_ws"
SCHEMA_NAME = "gold"

print("Notebook 7 — Route On-Time Performance")

In [0]:
df_matches_detailed = spark.sql("""
WITH scheduled_stops AS (
    SELECT 
        f.trip_id,
        f.stop_id,
        s.stop_lat,
        s.stop_lon,
        f.arrival_time AS scheduled_arrival,
        CAST(SUBSTRING(f.arrival_time, 1, 2) AS INT) AS sched_hour,
        f.route_short_name_display AS route_name,
        f.route_long_name
    FROM bc_transit_ws.gold.fact_trip_schedule f
    JOIN bc_transit_ws.silver.stops s ON f.stop_id = s.stop_id
    WHERE f.arrival_time IS NOT NULL
),
observations AS (
    SELECT
        trip_id,
        latitude,
        longitude,
        DATE(CONVERT_TIMEZONE('UTC', 'America/Vancouver', ingestion_timestamp)) AS obs_date,
        CONVERT_TIMEZONE('UTC', 'America/Vancouver', ingestion_timestamp) AS actual_ts
    FROM bc_transit_ws.gold.live_positions
),
matches AS (
    SELECT
        s.route_name,
        s.route_long_name,
        s.trip_id,
        s.stop_id,
        s.scheduled_arrival,
        o.obs_date,
        o.actual_ts,
        6371000 * 2 * ASIN(SQRT(
            POWER(SIN(RADIANS(o.latitude - s.stop_lat) / 2), 2) +
            COS(RADIANS(s.stop_lat)) * COS(RADIANS(o.latitude)) *
            POWER(SIN(RADIANS(o.longitude - s.stop_lon) / 2), 2)
        )) AS distance_meters,
        ROW_NUMBER() OVER (
            PARTITION BY s.trip_id, s.stop_id, o.obs_date
            ORDER BY 6371000 * 2 * ASIN(SQRT(
                POWER(SIN(RADIANS(o.latitude - s.stop_lat) / 2), 2) +
                COS(RADIANS(s.stop_lat)) * COS(RADIANS(o.latitude)) *
                POWER(SIN(RADIANS(o.longitude - s.stop_lon) / 2), 2)
            )) ASC
        ) AS rn
    FROM scheduled_stops s
    JOIN observations o ON s.trip_id = o.trip_id
    WHERE s.sched_hour IN (7, 12, 17)
    AND ABS(
        (UNIX_TIMESTAMP(o.actual_ts) - 
         UNIX_TIMESTAMP(CONCAT(o.obs_date, ' ', LPAD(TRIM(s.scheduled_arrival), 8, '0'))))
        / 60.0
    ) <= 30
)
SELECT 
    route_name,
    route_long_name,
    trip_id,
    stop_id,
    obs_date,
    scheduled_arrival,
    actual_ts AS actual_arrival,
    ROUND(distance_meters, 1) AS distance_meters,
    ROUND(
    (UNIX_TIMESTAMP(actual_ts) - 
     UNIX_TIMESTAMP(CONCAT(obs_date, ' ', LPAD(TRIM(scheduled_arrival), 8, '0')))
    ) / 60.0,
    2
) AS delay_minutes,
CASE 
    WHEN (
        (UNIX_TIMESTAMP(actual_ts) - 
         UNIX_TIMESTAMP(CONCAT(obs_date, ' ', LPAD(TRIM(scheduled_arrival), 8, '0')))
        ) / 60.0
    ) BETWEEN -1 AND 5 
    THEN 1 ELSE 0 
END AS is_on_time
FROM matches
WHERE rn = 1
AND distance_meters <= 50
""")

print(f"Detailed matches: {df_matches_detailed.count():,}")
df_matches_detailed.show(20, truncate=False)

In [0]:
DETAIL_PATH = f"{GOLD_BASE_PATH}stop_level_on_time/"

(df_matches_detailed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DETAIL_PATH)
)

(spark.read.format("delta").load(DETAIL_PATH).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.stop_level_on_time")
)

count = spark.read.format("delta").load(DETAIL_PATH).count()
print(f"Saved: bc_transit_ws.gold.stop_level_on_time ({count:,} rows)")

In [0]:
df_otp_summary = spark.sql("""
SELECT 
    route_name,
    route_long_name,
    COUNT(*) AS total_stop_observations,
    ROUND(AVG(ABS(delay_minutes)), 2) AS avg_delay_minutes,
    ROUND(STDDEV(delay_minutes), 2) AS delay_variance,
    ROUND(SUM(is_on_time) * 100.0 / COUNT(*), 2) AS on_time_pct
FROM bc_transit_ws.gold.stop_level_on_time
GROUP BY route_name, route_long_name
HAVING COUNT(*) >= 100
ORDER BY on_time_pct DESC
""")

OTP_PATH = f"{GOLD_BASE_PATH}route_on_time_performance/"

(df_otp_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(OTP_PATH)
)

(spark.read.format("delta").load(OTP_PATH).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.route_on_time_performance")
)

print(f"Saved: bc_transit_ws.gold.route_on_time_performance ({df_otp_summary.count()} routes)")
df_otp_summary.show(30, truncate=False)

In [0]:
df_wait_times = spark.sql("""
WITH consecutive_arrivals AS (
    SELECT
        route_name,
        route_long_name,
        stop_id,
        obs_date,
        actual_arrival,
        scheduled_arrival,
        LAG(actual_arrival) OVER (
            PARTITION BY route_name, stop_id, obs_date
            ORDER BY actual_arrival
        ) AS prev_actual_arrival,
        LAG(scheduled_arrival) OVER (
            PARTITION BY route_name, stop_id, obs_date
            ORDER BY actual_arrival
        ) AS prev_scheduled_arrival
    FROM bc_transit_ws.gold.stop_level_on_time
)
SELECT
    route_name,
    route_long_name,
    COUNT(*) AS total_gaps,
    ROUND(AVG(
        (UNIX_TIMESTAMP(actual_arrival) - UNIX_TIMESTAMP(prev_actual_arrival)) / 60.0
    ), 2) AS avg_actual_headway_min,
    ROUND(AVG(
        (UNIX_TIMESTAMP(CONCAT(obs_date, ' ', LPAD(TRIM(scheduled_arrival), 8, '0'))) - 
         UNIX_TIMESTAMP(CONCAT(obs_date, ' ', LPAD(TRIM(prev_scheduled_arrival), 8, '0')))) / 60.0
    ), 2) AS avg_scheduled_headway_min,
    ROUND(AVG(
        (UNIX_TIMESTAMP(actual_arrival) - UNIX_TIMESTAMP(prev_actual_arrival)) / 120.0
    ), 2) AS avg_passenger_wait_min
FROM consecutive_arrivals
WHERE prev_actual_arrival IS NOT NULL
AND (UNIX_TIMESTAMP(actual_arrival) - UNIX_TIMESTAMP(prev_actual_arrival)) / 60.0 <= 60
GROUP BY route_name, route_long_name
HAVING COUNT(*) >= 50
ORDER BY avg_passenger_wait_min ASC
""")

print(f"Routes with wait time analysis: {df_wait_times.count()}")
df_wait_times.show(30, truncate=False)

# Save to Gold
WAIT_PATH = f"{GOLD_BASE_PATH}route_passenger_wait_times/"

(df_wait_times.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(WAIT_PATH)
)

(spark.read.format("delta").load(WAIT_PATH).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.route_passenger_wait_times")
)

print(f"Saved: bc_transit_ws.gold.route_passenger_wait_times")

In [0]:
''' geographic delay hotpots'''

df_stop_delays = spark.sql("""
SELECT 
    o.stop_id,
    s.stop_name,
    s.stop_code,
    s.stop_lat,
    s.stop_lon,
    COUNT(*) AS observation_count,
    COUNT(DISTINCT o.route_name) AS routes_serving,
    ROUND(AVG(o.delay_minutes), 2) AS avg_delay_min,
    ROUND(SUM(CASE WHEN o.delay_minutes > 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_late_5min,
    ROUND(SUM(CASE WHEN o.delay_minutes > 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_late_10min,
    ROUND(SUM(o.is_on_time) * 100.0 / COUNT(*), 2) AS pct_on_time
FROM bc_transit_ws.gold.stop_level_on_time o
JOIN delta.`/Volumes/bc_transit_ws/gtfs/delta_volume/bronze/raw_stops/` s 
    ON o.stop_id = s.stop_id
WHERE o.delay_minutes IS NOT NULL
GROUP BY o.stop_id, s.stop_name, s.stop_code, s.stop_lat, s.stop_lon
HAVING COUNT(*) >= 50
ORDER BY avg_delay_min DESC
""")

print(f"Stops scored: {df_stop_delays.count()}")
print("\nWorst 20 stops by average delay:")
df_stop_delays.show(20, truncate=False)

# Save to Gold
STOP_DELAY_PATH = f"{GOLD_BASE_PATH}stop_delay_hotspots/"

(df_stop_delays.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(STOP_DELAY_PATH)
)

(spark.read.format("delta").load(STOP_DELAY_PATH).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.stop_delay_hotspots")
)

print(f"\nSaved: bc_transit_ws.gold.stop_delay_hotspots")

In [0]:
#route realibility at different peak hours time of the day

df_time_patterns = spark.sql("""
SELECT 
    o.route_name,
    o.route_long_name,
    CASE 
        WHEN HOUR(o.actual_arrival) IN (7, 8, 9)   THEN '7am'
        WHEN HOUR(o.actual_arrival) IN (12, 13, 14) THEN '12pm'
        WHEN HOUR(o.actual_arrival) IN (17, 18, 19) THEN '5pm'
        ELSE 'other'
    END AS fetch_window,
    COUNT(*) AS observation_count,
    ROUND(AVG(o.delay_minutes), 2) AS avg_delay_min,
    ROUND(SUM(o.is_on_time) * 100.0 / COUNT(*), 2) AS pct_on_time
FROM bc_transit_ws.gold.stop_level_on_time o
WHERE o.delay_minutes IS NOT NULL
GROUP BY o.route_name, o.route_long_name,
    CASE 
        WHEN HOUR(o.actual_arrival) IN (7, 8, 9)   THEN '7am'
        WHEN HOUR(o.actual_arrival) IN (12, 13, 14) THEN '12pm'
        WHEN HOUR(o.actual_arrival) IN (17, 18, 19) THEN '5pm'
        ELSE 'other'
    END
HAVING COUNT(*) >= 50
ORDER BY o.route_name, fetch_window
""")

print(f"Route-window combinations: {df_time_patterns.count()}")
df_time_patterns.show(30, truncate=False)

# Save to Gold
TIME_PATH = f"{GOLD_BASE_PATH}route_time_of_day_performance/"

(df_time_patterns.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TIME_PATH)
)

(spark.read.format("delta").load(TIME_PATH).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.route_time_of_day_performance")
)

print(f"Saved: bc_transit_ws.gold.route_time_of_day_performance")

In [0]:
spark.sql("""
SELECT 
    fetch_window,
    COUNT(DISTINCT route_name) AS routes,
    ROUND(AVG(avg_delay_min), 2) AS network_avg_delay_min,
    ROUND(AVG(pct_on_time), 2) AS network_avg_on_time_pct
FROM bc_transit_ws.gold.route_time_of_day_performance
WHERE fetch_window != 'other'
GROUP BY fetch_window
ORDER BY 
    CASE fetch_window WHEN '7am' THEN 1 WHEN '12pm' THEN 2 WHEN '5pm' THEN 3 END
""").show()

In [0]:
# Bus bunching with wait time for passengers

df_bunching_with_gaps = spark.sql("""
WITH consecutive_arrivals AS (
    SELECT
        route_name,
        route_long_name,
        stop_id,
        obs_date,
        actual_arrival,
        LAG(actual_arrival) OVER (
            PARTITION BY route_name, stop_id, obs_date
            ORDER BY actual_arrival
        ) AS prev_actual_arrival
    FROM bc_transit_ws.gold.stop_level_on_time
),
gaps AS (
    SELECT
        route_name,
        route_long_name,
        stop_id,
        obs_date,
        actual_arrival,
        (UNIX_TIMESTAMP(actual_arrival) - UNIX_TIMESTAMP(prev_actual_arrival)) / 60.0 AS gap_min
    FROM consecutive_arrivals
    WHERE prev_actual_arrival IS NOT NULL
)
SELECT
    route_name,
    route_long_name,
    COUNT(*) AS total_gaps,
    ROUND(AVG(gap_min), 2) AS avg_headway_min,
    ROUND(percentile_approx(gap_min, 0.5), 2) AS median_gap_min,
    ROUND(percentile_approx(gap_min, 0.9), 2) AS p90_gap_min,
    ROUND(percentile_approx(gap_min, 0.95), 2) AS p95_gap_min,
    ROUND(MAX(gap_min), 2) AS max_gap_min,
    SUM(CASE WHEN gap_min <= 2 THEN 1 ELSE 0 END) AS bunching_events,
    ROUND(SUM(CASE WHEN gap_min <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS bunching_rate_pct
FROM gaps
WHERE gap_min > 0 AND gap_min <= 60
GROUP BY route_name, route_long_name
HAVING COUNT(*) >= 100
ORDER BY p95_gap_min DESC
""")

print(f"Routes with full bunching analysis: {df_bunching_with_gaps.count()}")
df_bunching_with_gaps.show(30, truncate=False)

In [0]:
BUNCH_PATH = f"{GOLD_BASE_PATH}route_bus_bunching/"

(df_bunching_with_gaps.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BUNCH_PATH)
)

(spark.read.format("delta").load(BUNCH_PATH).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.route_bus_bunching")
)

print("Saved: bc_transit_ws.gold.route_bus_bunching")